# Analysis of drops velocity without cups

## 1. Documentation and Imports


Created on 11-02-2021

Author: Valentin Laplaud

This code analyses exepriments of calibration with the new drop setup (110 cm height) to obtain the drop velocity in function of drop size.

In [ ]:

# plotting stuff
import matplotlib as mpl
import matplotlib.pyplot as plt

# numbers handling
import numpy as np
import pandas as pd

# images handling
from skimage import io
from skimage.filters import threshold_otsu, gaussian
from skimage.measure import label, regionprops_table
from skimage.util import invert

# curve fitting
from scipy.optimize import curve_fit


## 2. Define analysis functions

### 2.1 Loading images and making kimographs

INPUTS :

Path to image stack

Spatial Scale computed before

TimeScale (frame rate)

In [ ]:
def ImagesToKymo(PathToStack, StackName, SpatialScale, TimeScale):
    
    stack = io.imread(PathToStack + '\\' + StackName) # get the tiff stack
        
    betterStack = ImgPreProc(stack)    
        
    StackL = stack.shape[0]
    imgH = stack.shape[1] # img height
    imgW = stack.shape[2] # img width
        
    if DebugPlots:
        plt.figure()
        plt.title(StackName)
        plt.imshow(stack[int(np.round(StackL/2))],cmap='gray')
    
    KymoH = betterStack[:, int(np.round(imgH/4)), :]
    
    KymoV = betterStack[:, :, int(np.round(imgW/2))]
    
    if DebugPlots:
        plt.figure()
        plt.title(StackName)
        plt.imshow(KymoH,cmap='gray')
        
        plt.figure()
        plt.title(StackName)
        plt.imshow(KymoV,cmap='gray')
        
    return(KymoH, KymoV)
    

### 2.2 Image preprocessing

Applying gaussian blur and ajusting LUT for better drop detection

In [ ]:
def ImgPreProc(img):
    
    finalImg = gaussian(img,2)
    
    return(finalImg)

### 2.3 Measuring drop diameter and speed from kymographs 

Threshold the kymographs obtained previously and extract the drop diameter and speed from the binary images

#### 2.3.1 Drop diameter from horizontal kymograph

Scale in px/mm

In [ ]:
def GetDropDiam(KymoH, Scale):
    
    BWimg = invert(KymoH > threshold_otsu(KymoH)) #threshold the image using the otsu method + inversion
  
    # extract properties from regions in the thresholded image
    label_img = label(BWimg)
    regions = regionprops_table(label_img, properties=('area',
                                                    'bbox'))
    
    propstable = pd.DataFrame(regions) # convert into dataframe

    idx = np.argmax(propstable.area) # index reprensting the biggest object    
   

    # compute diameter from bounding box edges 
    Diampx = propstable.at[idx, 'bbox-3'] - propstable.at[idx, 'bbox-1'] # in px    
    Diam = Diampx/Scale # in mm
    
    if DebugPlots:
        plt.figure()
        plt.imshow(BWimg,cmap='gray')
                

    return(Diam)
       

#### 2.3.2 Drop velocity from vertical kymograph

Scale in px/cm

In [ ]:
def GetDropVel(KymoV, Scale, FPS):
    
    BWimg = invert(KymoV > threshold_otsu(KymoV)) #threshold the image using the otsu method + inversion
  
    # find the white pixels in the image
    y, x = np.where(BWimg == True)

    # fit a line going through pixels positions
    fit = np.polyfit(x,y,1)

    # compute drop velocity from fit coefficients
    Velpx = 1/fit[0] # in px/px (dist/time)
    Vel = Velpx/Scale*FPS
    
    if DebugPlots:
        plt.figure()
        plt.scatter(x,y,marker='.')
        plt.plot(x,fit[1]+x*fit[0],'r')

        plt.figure(dpi = 200)
        plt.imshow(BWimg,cmap='gray')
        plt.plot(x,fit[1]+x*fit[0],'r',linewidth=1)
    
    return(Vel)

## 3. Fitting 

### 3.1 Fitting velocity and radius 

The function to fit is V(r) = r^(1/2) * a * tanh(r^(-1/2) * b) where a,b are parameters. This equation comes from the resolution of the differential equation of a sphere falling in air. The time T can be computed as (b * a / g) where g = 9.8 m/s²

It returns the fit parameters as well as plotable vectors to represent the fit

In [ ]:
def FitDropVel(V,R):

    def VofR(x,a,b): # x radius in mm, a,b constant depending on time
        return(np.sqrt(x)*a*np.tanh(b/np.sqrt(x)))
    
    popt, pcov = curve_fit(VofR, R, V)
    
    Rfit = np.linspace(np.min(R),np.max(R),30)
    
    Vfit = VofR(Rfit, *popt)
    
    return(Rfit, Vfit, popt)


## 4. Enter data and run

In [ ]:

DebugPlots = False # Display intermediate plots along the way

# Paths to data
P = r'd:\Users\Valentin Laplaud\PostDoc\Data\SplashCups\Manips\21.02.09_DropSpeedSide'

# List of stacks to analyse
StackList_2_3cm = ['2-3cm_1-1.tif','2-3cm_2-1.tif','2-3cm_2-3.tif','2-3cm_2-4.tif','2-3cm_2-5.tif', 
             '2-3cm_3.tif', '2-3cm_4.tif', '2-3cm_5.tif', '2-3cm_6.tif', '2-3cm_8.tif',
             '2-3cm_9.tif', '2-3cm_10.tif', '2-3cm_11.tif', '2-3cm_12.tif', '2-3cm_13.tif', '2-3cm_14.tif',
             '2-3cm_15-1.tif','2-3cm_15-2.tif','2-3cm_15-3.tif','2-3cm_15-4.tif','2-3cm_15-5.tif','2-3cm_15-6.tif',
             '2-3cm_15-7.tif','2-3cm_15-8.tif','2-3cm_15-9.tif','2-3cm_15-10.tif','2-3cm_15-11.tif',
             '2-3cm_15-12.tif','2-3cm_15-13.tif']


StackList_9_10cm = ['9-10cm_1.tif','9-10cm_2.tif','9-10cm_3.tif','9-10cm_4.tif','9-10cm_5.tif','9-10cm_6.tif'
                   ,'9-10cm_7.tif','9-10cm_8.tif','9-10cm_9.tif','9-10cm_10.tif','9-10cm_11.tif','9-10cm_12.tif'
                   ,'9-10cm_13.tif','9-10cm_14-1.tif','9-10cm_14-2.tif','9-10cm_14-3.tif','9-10cm_14-4.tif'
                   ,'9-10cm_14-5.tif','9-10cm_14-6.tif','9-10cm_14-7.tif','9-10cm_14-8.tif','9-10cm_14-9.tif'
                   ,'9-10cm_14-10.tif','9-10cm_14-11.tif','9-10cm_14-12.tif','9-10cm_14-13.tif','9-10cm_14-14.tif'
                   ,'9-10cm_14-15.tif','9-10cm_14-18.tif','9-10cm_14-19.tif'
                   ,'9-10cm_14-20.tif','9-10cm_14-21.tif','9-10cm_14-22.tif']

StackList_Cups = ['Speed_C1_1.tif','Speed_C1_2-1.tif','Speed_C1_3-1.tif','Speed_C2_1-1.tif','Speed_C2_2-1.tif'
                 ,'Speed_C2_3-1.tif','Speed_C2_4-1.tif','Speed_C2_5-1.tif','Speed_C2_6-1.tif'
                 ,'Speed_C2_7-1.tif','Speed_C2_8-1.tif','Speed_C2_9-1.tif']

FPS = 10000 # Framerate

# initializing drops properties dataframe
DropsProps = pd.DataFrame(data=None,columns=['Drop','MeasureHeight (cm)','Diameter (mm)','Velocity (m/s)']) 

# Data for 2-3 cm height measurements
Scale = 454.4 # Spatial scale (px/cm)

for s in StackList_2_3cm:
    
    KymoH, KymoV = ImagesToKymo(P, s, Scale, FPS)
    
    Diam = GetDropDiam(KymoH, Scale/10) # in mm
    
    Vel = GetDropVel(KymoV, Scale*100, FPS) # in cm/s
    
    # creating dataframe for current drop
    dropprops = pd.DataFrame(data=[[s, 2.5, Diam, Vel]],columns=['Drop','MeasureHeight (cm)','Diameter (mm)','Velocity (m/s)']) 

    
    DropsProps = DropsProps.append(dropprops) # adding to global dataframe

# Data for 9-10 cm height measurements
Scale = 479.8 # Spatial scale (px/cm)

for s in StackList_9_10cm:
    
    KymoH, KymoV = ImagesToKymo(P, s, Scale, FPS)
    
    Diam = GetDropDiam(KymoH, Scale/10) # in mm
    
    Vel = GetDropVel(KymoV, Scale*100, FPS) # in m/s
    
    # creating dataframe for current drop
    dropprops = pd.DataFrame(data=[[s, 9.5, Diam, Vel]],columns=['Drop','MeasureHeight (cm)','Diameter (mm)','Velocity (m/s)']) 

    
    DropsProps = DropsProps.append(dropprops) # adding to global dataframe
    
# Data of impact in cups
Scale = 445.1 # Spatial scale (px/cm)

for s in StackList_Cups:
    
    KymoH, KymoV = ImagesToKymo(P, s, Scale, FPS)
    
    Diam = GetDropDiam(KymoH, Scale/10) # in mm
    
    Vel = GetDropVel(KymoV, Scale*100, FPS) # in m/s
    
    # creating dataframe for current drop
    dropprops = pd.DataFrame(data=[[s, 0, Diam, Vel]],columns=['Drop','MeasureHeight (cm)','Diameter (mm)','Velocity (m/s)']) 

    
    DropsProps = DropsProps.append(dropprops) # adding to global dataframe    
    
# getting velocity and drop radius for first observation height (2-3 cm)
DropsProps23 = DropsProps[DropsProps['MeasureHeight (cm)'] == 2.5]
V23 = DropsProps23['Velocity (m/s)'].values
R23 = DropsProps23['Diameter (mm)'].values

# getting velocity and drop radius for second observation height (9-10 cm)
DropsProps910 = DropsProps[DropsProps['MeasureHeight (cm)'] == 9.5]
V910 = DropsProps910['Velocity (m/s)'].values
R910 = DropsProps910['Diameter (mm)'].values


# Fit velocity and get associated time

R23fit, V23fit, param23 = FitDropVel(V23,R23)
R910fit, V910fit,  param910 = FitDropVel(V910,R910)

# Computing time from fit parameters
T23 = param23[1]*param23[0]/9.8
T910 = param910[1]*param910[0]/9.8

# plots

fig = plt.figure(dpi=150)
plt.axis()
ax1 = plt.gca()
DropsProps[DropsProps['MeasureHeight (cm)'] == 2.5].plot('Diameter (mm)', 'Velocity (m/s)',
                                                         kind='scatter', xlabel = 'Diameter (mm)', 
                                                         ylabel='Velocity (m/s)', ax = ax1,c='r')

DropsProps[DropsProps['MeasureHeight (cm)'] == 9.5].plot('Diameter (mm)', 'Velocity (m/s)', 
                                                         kind='scatter', xlabel = 'Diameter (mm)', 
                                                         ylabel='Velocity (m/s)', ax = ax1)

DropsProps[DropsProps['MeasureHeight (cm)'] == 0].plot('Diameter (mm)', 'Velocity (m/s)', 
                                                         kind='scatter', xlabel = 'Diameter (mm)', 
                                                         ylabel='Velocity (m/s)', ax = ax1,c='g')

plt.plot(R23fit,V23fit,'+-m')
plt.plot(R910fit,V910fit,'*-c')

plt.legend(['Fit 2-3 cm, T = ' + str(round(T23*100)/100) + ' sec',
            'Fit 9-10 cm, T = ' + str(round(T910*100)/100) + ' sec',
            'Data 2-3cm','Data 9-10cm', 'Data cups'])

plt.savefig(P + '\\' + 'DropSpeed.png')


# Test Zone